In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
import numpy as np
import pandas as pd
import random
import math
import json
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from tqdm import tqdm

# shared, de-duplicated implementations for the semi-supervised family
from src.semi import (
    TextDataset, DataFactory, mmd_loss,
    Classifier, BERTClassifier, RoBERTaClassifier, LSTM_BERT_Classifier,
    SemiMain,
)


In [2]:
class Main(SemiMain):

    def train_on_cls_embeddings(self, epochs=15, batch_size=32, lr=1e-5, use_mixup=True):
        """
        Fine-tunes only the classifier head using precomputed [CLS] embeddings of new data set.
        """
        device = self.device
        min_loss = math.inf
        patience = 3
        self.model.to(device)
        self.model.train()

        # Freeze non-head parts
        for p in self.model.bert.parameters():
            p.requires_grad = False
        for p in self.model.proj.parameters():
            p.requires_grad = False

        train_loader = self.second_trainloader
        val_loader = self.second_valloader

        # Train only fc and classifier layers
        params = list(self.model.fc.parameters()) + list(self.model.classifier.parameters())
        optimizer = torch.optim.Adam(params, lr=lr)
        criterion = nn.CrossEntropyLoss()

        def one_hot(labels, num_classes):
            return torch.nn.functional.one_hot(labels, num_classes=num_classes).float()
        # Mixup oversampling
        def mixup_embeddings(x, y, alpha=0.2):
            lam = np.random.beta(alpha, alpha)
            index = torch.randperm(x.size(0)).to(x.device)
            mixed_x = lam * x + (1 - lam) * x[index]
            mixed_y = lam * y + (1 - lam) * y[index]
            return mixed_x, mixed_y

        for epoch in range(epochs):
            self.model.train()
            epoch_loss, epoch_acc, epoch_f1 = [], [], []

            for cls_vec, y, _ in tqdm(train_loader):
                cls_vec, y = cls_vec.to(device), y.to(device)

                optimizer.zero_grad()

                if use_mixup:
                    y_onehot = one_hot(y, self.configs.num_classes).to(device)
                    cls_vec, y_mix = mixup_embeddings(cls_vec, y_onehot)
                    cls_out = self.model.fc(cls_vec)
                    logits = self.model.classifier(cls_out + cls_vec)
                    loss = torch.sum(-y_mix * torch.nn.functional.log_softmax(logits, dim=1), dim=1).mean()
                    pred = torch.argmax(logits, dim=1)
                    true_y = torch.argmax(y_mix, dim=1)
                else:
                    cls_out = self.model.fc(cls_vec)
                    logits = self.model.classifier(cls_out + cls_vec)
                    loss = criterion(logits, y)
                    pred = torch.argmax(logits, dim=1)
                    true_y = y

                loss.backward()
                optimizer.step()

                epoch_loss.append(loss.item())
                epoch_acc.append(self.metric1(pred.cpu(), true_y.cpu()))
                epoch_f1.append(self.metric2(pred.cpu(), true_y.cpu(), average='macro'))

            print(f"[Head Epoch {epoch + 1}] Loss: {np.mean(epoch_loss):.4f}, "
                f"Acc: {np.mean(epoch_acc):.4f}, F1: {np.mean(epoch_f1):.4f}")

            val_loss = self.second_validation()
            self.model.train()

            if val_loss < min_loss:
                min_loss = val_loss
                self.__save__()
                patience = 3
            else:
                patience -= 1

            if patience == 0:
                print("Early stopping triggered.")
                break
        self.__load__()


In [3]:
class Config:
    lambda_mmd = 0.001
    name = 'BERT_mixup_semi'
    embed_dim = 256
    lstm_hidden_dim = 64
    num_heads = 16
    hidden_dim = 128
    num_layers = 2
    num_classes = 2
    path1 = '../data/raw/domain1_train_data.json'
    path2 = '../data/raw/domain2_train_data.json'
    test_path = '../data/raw/test_data.json'
    num_epochs = 15
    batch_size1 = 32
    batch_size2 = 32
    seed = 42
    tau=1
    save_path = "../results/mixup_pred.csv"

configs = Config()
main = Main(configs)

In [4]:
'''get raw data from data loader'''
def extract_raw_data(dataset):
    result = []
    for i in range(len(dataset)):
        text, label, idx = dataset[i]
        mask = (text != 0).long()
        result.append((text, label, idx))
    return result

In [5]:
def build_dataloader_from_pseudo(data_list, batch_size, collate_fn):
    class PseudoDataset(torch.utils.data.Dataset):
        def __init__(self, data):
            self.data = data  # (text, mask, label, id)

        def __len__(self):
            return len(self.data)

        def __getitem__(self, idx):
            text, label, idx_ = self.data[idx]
            return text, label, idx_ 

    dataset = PseudoDataset(data_list)
    loader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
    return loader


In [6]:
class SecondPhaseDataset(Dataset):
    def __init__(self, features, labels, ids):
        self.features = torch.tensor(features, dtype=torch.float)
        self.labels = torch.tensor(labels, dtype=torch.long)
        self.ids = ids

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx], torch.tensor(self.ids[idx], dtype=torch.long)


In [7]:
main.train()

0it [00:00, ?it/s]

1it [00:00,  3.53it/s]

2it [00:00,  3.67it/s]

3it [00:00,  3.70it/s]

4it [00:01,  3.71it/s]

...

7it [00:00,  8.95it/s]

8it [00:00,  9.13it/s]

9it [00:01,  8.87it/s]

10it [00:01,  9.05it/s]

Validation Loss: 0.2974, Accuracy: 0.9141, F1: 0.8810


In [8]:

from torch.utils.data import Dataset, DataLoader
# collect pseudo data
pseudo_data_1, pseudo_data_2, val_data_1, val_data_2 = main.collect_pseudo_labels(threshold=0.99)
train_data_1 = extract_raw_data(main.trainloader_1.dataset)
train_data_2 = extract_raw_data(main.trainloader_2.dataset)

# create dataloader for new data set
combined_train_data_1 = train_data_1 + pseudo_data_1
combined_train_data_2 = train_data_2 + pseudo_data_2
temp_loader_1 = build_dataloader_from_pseudo(combined_train_data_1, batch_size=32, collate_fn=main.data2.collate_fn)
temp_loader_2 = build_dataloader_from_pseudo(combined_train_data_2, batch_size=32, collate_fn=main.data2.collate_fn)

# embeddings
feat_1, label_1, id_1 = main.model.extract_cls_from_tokens(temp_loader_1, main.device)
feat_2, label_2, id_2 = main.model.extract_cls_from_tokens(temp_loader_2, main.device)

features = torch.cat([feat_1, feat_2])
labels = torch.cat([label_1, label_2])
ids = id_1 + id_2

dataset = SecondPhaseDataset(features.numpy(), labels.numpy(), ids)
main.second_trainloader = DataLoader(dataset, batch_size=main.configs.batch_size1, shuffle=True)

val_data = val_data_1 + val_data_2
# Build validation loaders from remaining validation data
val_loader = build_dataloader_from_pseudo(val_data, batch_size=main.configs.batch_size1, collate_fn=main.data2.collate_fn)

# Assign to main class
main.second_valloader = val_loader

main.train_on_cls_embeddings()  # Phase 2 training with updated loaders



0it [00:00, ?it/s]

1it [00:00,  9.62it/s]

2it [00:00,  9.56it/s]

4it [00:00, 10.08it/s]

6it [00:00, 10.16it/s]

...

 83%|████████▎ | 120/145 [00:00<00:00, 144.09it/s]

 94%|█████████▍| 137/145 [00:01<00:00, 150.53it/s]

100%|██████████| 145/145 [00:01<00:00, 129.83it/s]

[Head Epoch 4] Loss: 0.0871, Acc: 0.9875, F1: 0.9689


[Second Val] Loss: 0.1869, Acc: 0.9955, F1: 0.9954
Early stopping triggered.


In [9]:
'''final validation on all data, including train and validation'''
train_data_1 = extract_raw_data(main.trainloader_1.dataset)
train_data_2 = extract_raw_data(main.trainloader_2.dataset)
val_data_1 = extract_raw_data(main.valloader_1.dataset)
val_data_2 = extract_raw_data(main.valloader_2.dataset)
full_data = train_data_1 + train_data_2 + val_data_1 + val_data_2
full_loader = build_dataloader_from_pseudo(full_data, configs.batch_size2, main.data2.collate_fn)
def evaluate_model(model, dataloader, criterion, device, source_name="all"):
    model.eval()
    all_preds, all_labels, all_ids = [], [], []
    misclassified = []

    total_loss = 0

    with torch.no_grad():
        for x, x_mask, y, ids in dataloader:
            x, x_mask, y = x.to(device), x_mask.to(device), y.to(device)
            outputs, _ = model(x, x_mask)
            loss = criterion(outputs, y)
            total_loss += loss.item()
            preds = torch.argmax(outputs, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y.cpu().numpy())
            all_ids.extend(ids.cpu().numpy())

            for i in range(len(y)):
                true_label = y[i].item()
                pred_label = preds[i].item()
                sample_id = ids[i].item()
                if true_label != pred_label:
                    misclassified.append((sample_id, true_label, pred_label, source_name))

    from sklearn.metrics import accuracy_score, f1_score
    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average='macro')

    print(f"\nEvaluation on {source_name} — Loss: {total_loss:.4f}, Accuracy: {acc:.4f}, F1: {f1:.4f}")
    print(f"Misclassified Samples: {len(misclassified)}")
    return

evaluate_model(main.model, full_loader, main.criterion, main.device)




Evaluation on all — Loss: 11.4726, Accuracy: 0.9782, F1: 0.9498
Misclassified Samples: 131


In [10]:
# main.test()